## Introduction

This chapter looks smaller than chunking or embeddings, but tokenization sits underneath almost every LLM and RAG operation.

It affects:

- chunk size
- context-window limits
- embedding limits
- prompt cost
- multilingual RAG
- retrieval context size
- latency
- truncation
- generation limits

The key idea is:

> **LLMs do not directly process words or characters. They process sequences of numerical token IDs.**

## Basic Definitions in Simple Words

| Term | Meaning in simple words |
|---|---|
| **Token** | A small piece of text the model reads: a word, part of a word, a space plus a word, or punctuation. |
| **Token ID** | The number that stands for a token in the model's vocabulary, e.g. `" love"` may be 3021. |
| **Tokenizer** | The program that cuts text into tokens and turns them into token IDs (and back). |
| **Vocabulary** | The fixed list of all tokens a tokenizer knows, often 30,000 to 200,000 entries. |
| **Subword** | A piece smaller than a word, like `un`, `believ`, `able`. Lets rare words be built from common parts. |
| **BPE (Byte Pair Encoding)** | Learns a vocabulary by repeatedly merging the most frequent neighbouring pair. Used by GPT models. |
| **WordPiece** | A subword method used by BERT; pieces that continue a word start with `##`. |
| **SentencePiece** | A toolkit that tokenizes raw text directly (spaces included) with BPE or Unigram; common for multilingual models. |
| **Byte-level tokenizer** | Works on UTF-8 bytes, so any text can be encoded and nothing is ever "unknown". |
| **Special tokens** | Reserved tokens with a job, like `[CLS]`, `[SEP]`, `<eos>`, `[PAD]`. |
| **Padding / attention mask** | Filler tokens that make inputs the same length, and the 0/1 mask that tells the model to ignore them. |
| **Context window** | The maximum number of tokens a model can read plus write in one request. |
| **Token budget** | How many of those tokens you give to system prompt, history, query, retrieved chunks and the answer. |
| **Truncation** | Cutting off tokens past the model's limit. Often silent: the cut text is simply ignored. |
| **Compression ratio** | Characters (or bytes) per token. Higher means fewer tokens for the same text. |

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    T["Text"] --> TK["Tokenizer"] --> TO["Tokens"] --> ID["Token IDs"] --> M["Model"]
    ID -.->|"count"| B(["Chunk size, context<br/>window, cost"])
    style B fill:#FEF3C7,stroke:#D97706
```

## 1. What exactly is a token?

> [!TIP]
> **In simple words:** A token is the unit an LLM reads. It is often a word, but can be part of a word, a space plus a word, or punctuation.

Consider:

```text
I love artificial intelligence.
```

A human thinks:

```text
I
love
artificial
intelligence
```

Four words.

But an LLM may internally see something conceptually like:

```text
"I"
" love"
" artificial"
" intelligence"
"."
```

or perhaps:

```text
"I"
" love"
" artific"
"ial"
" intelligence"
"."
```

depending on the tokenizer.

Each piece is a **token**.

The model ultimately receives numerical IDs:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    A["Text<br/>I love AI"] -->|"tokenizer"| B["Tokens<br/>I, love, AI"] -->|"vocabulary<br/>lookup"| C["Token IDs<br/>40, 3021, 15592"] --> D["LLM"]
```

So:

```math
\text{Text} \rightarrow \text{Tokens} \rightarrow \text{Token IDs} \rightarrow \text{Model}
```

## 2. Token ≠ word

This is the first important interview concept.

Consider:

```text
cat
```

Could be one token.

But:

```text
electroencephalography
```

could become several subword tokens.

Likewise:

```text
2026/1542
```

might get split into several pieces.

Therefore:

```math
\text{Number of words} \neq \text{Number of tokens}
```

And also:

```math
\text{Number of characters} \neq \text{Number of tokens}
```

## 3. Why don't LLMs simply tokenize by words?

> [!TIP]
> **In simple words:** Too many possible words and new words every day. Subwords keep the vocabulary small and still spell any word.

Imagine vocabulary contains every English word.

You immediately encounter:

```text
walk
walks
walked
walking
```

and:

```text
RAG
GraphRAG
AgenticRAG
```

and names:

```text
Ajay
Vaibhav
OpenAI
```

and millions of rare strings.

A word-level vocabulary becomes enormous.

Even worse:

> What happens when the model sees a completely new word?

Example:

```text
HyperQuantumRAG2027
```

If it's absent from the vocabulary, a pure word tokenizer has a problem.

## 4. Character tokenization?

Other extreme:

```text
intelligence
```

becomes:

```text
i
n
t
e
l
l
i
g
e
n
c
e
```

This solves unknown words because almost every word can be represented using characters.

But sequences become extremely long.

Suppose a document contains:

```math
100,000 \text{ characters}
```

Character tokenization might produce nearly:

```math
100,000 \text{ tokens}
```

Very inefficient.

## 5. The middle ground: subword tokenization

> [!TIP]
> **In simple words:** Common words stay whole; rare words are built from common pieces like un + believ + able.

Modern language models typically use **subword-based tokenization** or related schemes.

Instead of:

```text
unbelievable
```

as one unknown word or 12 separate characters, you might get something like:

```text
un
believ
able
```

Conceptually:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    C["Character-level<br/>u n b e l i e v a b l e"] -->|"too granular"| X1((" "))
    W2["Word-level<br/>unbelievable"] -->|"might be rare"| X2((" "))
    S["Subword-level<br/>un + believ + able"] -->|"good compromise"| X3((" "))
    style S fill:#DCFCE7,stroke:#16A34A
    style X1 fill:none,stroke:none
    style X2 fill:none,stroke:none
    style X3 fill:none,stroke:none
```

This provides:

```text
manageable vocabulary
+
ability to represent unseen words
+
reasonable sequence length
```

## 6. Why tokenization matters specifically in RAG

> [!TIP]
> **In simple words:** Every limit in RAG is counted in tokens: chunk size, embedding input, context window and price.

Recall 04.01 Document Chunking:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    n0["Document"] --> n1["Chunk"] --> n2["Embed"] --> n3["Retrieve"]
```

Suppose we say:

```text
chunk_size = 500
```

500 what?

Characters?

Words?

Tokens?

For LLM systems, tokens are often the most meaningful unit because both embedding models and generation models have token limits.

## 7. RAG token flow

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    n0["Document"] --> n1["Tokenizer"] --> n2["Tokens"] --> n3["Chunk into<br/>500-token units"] --> n4["Embedding model"]
```

At query time:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    S["System<br/>prompt"] --> T["Tokenizer"]
    H["Conversation<br/>history"] --> T
    R["Retrieved<br/>chunks"] --> T
    Q["User<br/>query"] --> T
    T --> N["Total input<br/>tokens"] --> L["LLM"]
```

You must satisfy:

```math
T_{\text{input}}+T_{\text{output}} \leq T_{\text{context}}
```

## 8. Context-window mathematics

> [!TIP]
> **In simple words:** The window is shared: system prompt, history, query, retrieved chunks and the answer must all fit.

Suppose model context window:

```math
T_{\text{context}}=32,000
```

You reserve:

```math
T_{\text{output}}=4,000
```

System prompt:

```math
T_{\text{system}}=1,500
```

Conversation history:

```math
T_{\text{history}}=5,000
```

User query:

```math
T_{\text{query}}=500
```

Available retrieval budget:

```math
T_{\text{retrieval}} = T_{\text{context}} - T_{\text{output}} - T_{\text{system}} - T_{\text{history}} - T_{\text{query}}
```

Therefore:

```math
T_{\text{retrieval}} = 32,000 - 4,000 - 1,500 - 5,000 - 500 =21,000
```

So theoretically you have: $21,000$

tokens for retrieved evidence.

## 9. If each chunk is 500 tokens

Then theoretically:

```math
\frac{21,000}{500}=42
```

chunks could fit.

But should you retrieve 42?

Usually no.

Why?

Because:

```text
more context
≠
better answer
```

You may create:

- irrelevant context
- redundancy
- increased latency
- increased cost
- lost-in-the-middle problems

So token budget gives you a **maximum**, not the optimal Top-K.

## 10. Token vocabulary

A tokenizer has a vocabulary:

```math
V=\{t_1,t_2,\ldots,t_n\}
```

Each token gets an integer ID.

Example:

```text
Token            ID

"the"            1032
"ing"            849
" RAG"           29173
"."              13
```

Conceptually:

```math
\text{token} \rightarrow \text{integer}
```

The neural network doesn't receive:

```text
"RAG"
```

It receives something like: $29173$

Then that ID is mapped to an embedding vector.

## 11. Token ID → embedding

Suppose vocabulary size: $|V|=100,000$

Embedding dimension: $d=4096$

There is conceptually an embedding matrix:

```math
E \in \mathbb{R}^{100000\times4096}
```

Token ID: $i=29173$

selects: $E_{29173}$

which is a vector of dimension 4096.

So:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    n0["Text: "RAG""] --> n1["Token ID 29173"] --> n2["Embedding lookup"] --> n3["Vector [0.12,<br/>-0.71, 0.33, ...]"]
```

Do not confuse this internal **token embedding** with the document embeddings used for RAG retrieval. They are related ideas but different parts of the system.

## 12. Main tokenizer families

For interviews, know these:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart TB
    T["Tokenization"] --> W["Word-level"]
    T --> C["Character-level"]
    T --> S["Subword"]
    S --> B["BPE"]
    S --> WP["WordPiece"]
    S --> SP["SentencePiece-<br/>based approaches"]
    style S fill:#DCFCE7,stroke:#16A34A
```

The most important ones to understand are:

1. **BPE**
2. **WordPiece**
3. **SentencePiece**

## 13. Byte Pair Encoding — BPE

> [!TIP]
> **In simple words:** Start from characters, find the most frequent neighbouring pair, glue it into one token, repeat.

BPE is extremely important.

The intuition:

> Start with small units and repeatedly merge frequently occurring adjacent pairs.

Consider corpus:

```text
low
lower
lowest
```

Initially:

```text
l o w
l o w e r
l o w e s t
```

Count adjacent pairs.

Possible frequencies:

```text
(l,o)    3
(o,w)    3
(w,e)    2
(e,r)    1
(e,s)    1
```

Most frequent pair:

```text
(l,o)
```

Merge:

```text
lo
```

Now:

```text
lo w
lo w e r
lo w e s t
```

Next frequent:

```text
(lo,w)
```

Merge:

```text
low
```

Now:

```text
low
low e r
low e s t
```

The tokenizer has learned a useful reusable subword:

```text
low
```

## 14. BPE intuition visually

Start:

```text
l o w e r
```

Merge:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    n0["l + o"] --> n1["lo w e r"]
```

Next:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    n0["lo + w"] --> n1["low e r"]
```

Eventually:

```text
low
lower
```

depending on vocabulary size and learned merge rules.

## 15. Why BPE works

Suppose vocabulary learned:

```text
play
ing
er
ed
```

Then it can represent:

```text
playing
player
played
```

using combinations:

```text
play + ing
play + er
play + ed
```

Instead of storing each complete word separately.

## 16. BPE training concept

Given training corpus: $C$

Start vocabulary from basic symbols.

Then repeatedly:

1. calculate adjacent-pair frequencies
2. identify most frequent pair
3. merge it
4. add merged symbol to vocabulary

until:

```math
|V|=V_{\text{target}}
```

## 17. Simple BPE implementation from scratch

Let's make a tiny educational version.

### Step 1 — Get adjacent pairs

In [1]:
# Define a function that returns adjacent token pairs.
def get_adjacent_pairs(tokens):

    # Create an empty list for token pairs.
    pairs = []

    # Iterate until the second-last token.
    for index in range(len(tokens) - 1):

        # Create a pair from neighboring tokens.
        pair = (
            tokens[index],
            tokens[index + 1],
        )

        # Store the pair.
        pairs.append(pair)

    # Return all adjacent pairs.
    return pairs

## 18. Count pair frequencies

In [2]:
# Import Counter for frequency counting.
from collections import Counter

# Define a function that counts adjacent pairs across words.
def count_pair_frequencies(tokenized_words):

    # Create a frequency counter.
    pair_counts = Counter()

    # Process every tokenized word.
    for tokens in tokenized_words:

        # Extract adjacent pairs.
        pairs = get_adjacent_pairs(tokens)

        # Add pair frequencies.
        pair_counts.update(pairs)

    # Return pair counts.
    return pair_counts

Example:

In [3]:
# Create a tiny corpus.
words = [
    list("low"),
    list("lower"),
    list("lowest"),
]

# Count adjacent pair frequencies.
pair_counts = count_pair_frequencies(words)

# Display frequencies.
print(pair_counts)

Counter({('l', 'o'): 3, ('o', 'w'): 3, ('w', 'e'): 2, ('e', 'r'): 1, ('e', 's'): 1, ('s', 't'): 1})


## 19. Merge a selected pair

In [4]:
# Define a function that merges one selected token pair.
def merge_pair(tokens, pair_to_merge):

    # Create a list for merged output.
    merged_tokens = []

    # Start at the first token.
    index = 0

    # Continue until all tokens are processed.
    while index < len(tokens):

        # Check whether the current and next token match the pair.
        if (
            index < len(tokens) - 1
            and tokens[index] == pair_to_merge[0]
            and tokens[index + 1] == pair_to_merge[1]
        ):

            # Combine the two tokens.
            merged_token = tokens[index] + tokens[index + 1]

            # Store the merged token.
            merged_tokens.append(merged_token)

            # Skip both original tokens.
            index += 2

        else:

            # Keep the current token unchanged.
            merged_tokens.append(tokens[index])

            # Move to the next token.
            index += 1

    # Return the transformed token sequence.
    return merged_tokens

## 20. Perform one BPE iteration

In [5]:
# Count pair frequencies.
pair_counts = count_pair_frequencies(words)

# Find the most frequent pair.
most_common_pair = pair_counts.most_common(1)[0][0]

# Create a list for updated words.
updated_words = []

# Merge the selected pair inside every word.
for tokens in words:

    # Apply the merge operation.
    merged_tokens = merge_pair(
        tokens,
        most_common_pair,
    )

    # Store the updated representation.
    updated_words.append(merged_tokens)

# Display the selected merge.
print("Merged pair:", most_common_pair)

# Display updated words.
print(updated_words)

Merged pair: ('l', 'o')
[['lo', 'w'], ['lo', 'w', 'e', 'r'], ['lo', 'w', 'e', 's', 't']]


Repeat the process and BPE gradually learns larger units.

## 21. Real BPE is more sophisticated

Our implementation is educational.

Production tokenizers have additional concerns:

```text
byte handling
Unicode
whitespace
special tokens
pre-tokenization
vocabulary limits
merge ordering
efficient implementations
```

But the fundamental intuition remains:

```math
\text{frequent adjacent units} \rightarrow \text{merged token}
```

## 22. WordPiece

> [!TIP]
> **In simple words:** Like BPE, but pieces that continue a word are marked with ##, so the model knows they belong to the previous piece.

WordPiece is famously associated with BERT-family tokenization.

It is also subword-based.

Example conceptually:

```text
playing
```

could become:

```text
play
##ing
```

Here:

```text
##ing
```

means roughly:

> this subword continues an existing word.

Example:

```text
unhappiness
```

might conceptually become:

```text
un
##happi
##ness
```

Exact tokenization depends on the vocabulary.

## 23. BPE vs WordPiece

They are similar in spirit but differ in how merges/vocabulary are selected.

High-level interview distinction:

### BPE

Generally merges based strongly on frequent adjacent pairs.

### WordPiece

Uses a likelihood/scoring-based criterion to build useful subword vocabulary.

Don't spend interview time deriving obscure implementation differences unless explicitly asked.

Main point:

```text
Both solve the rare/unknown word problem
using subword units.
```

## 24. SentencePiece

> [!TIP]
> **In simple words:** It treats the space as a normal character, so it works for languages that do not use spaces.

SentencePiece is slightly different conceptually.

Many tokenizers first separate text into words using spaces.

SentencePiece can train directly on raw text.

This is useful for languages where whitespace isn't a reliable word separator.

It commonly uses algorithms such as:

```text
BPE
or
Unigram Language Model
```

Important distinction:

> **SentencePiece is a tokenization framework, not one single tokenization algorithm.**

That is a useful interview point.

## 25. Why SentencePiece matters for multilingual models

Consider Chinese:

```text
人工智能
```

Word boundaries aren't represented like English spaces.

Japanese has similar challenges.

A tokenizer that assumes:

```text
split on spaces
```

is not ideal.

SentencePiece can operate directly over raw Unicode sequences.

## 26. Byte-level tokenization

Some modern tokenizers operate at byte/subword level.

An advantage:

> Any input can theoretically be represented without requiring an unknown token for arbitrary Unicode strings.

Conceptually:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    n0["Unicode text"] --> n1["UTF-8 bytes"] --> n2["Byte-based symbols"] --> n3["Merge frequent<br/>sequences"]
```

This gives robust coverage.

## 27. The unknown-token problem

Older word/subword systems may use:

```text
[UNK]
```

meaning unknown.

Example:

```text
SuperHyperRAGX
```

could theoretically become:

```text
[UNK]
```

This loses information.

Byte-level approaches reduce the need for such fallback behavior because arbitrary input can be represented via bytes.

## 28. Special tokens

Models also use special control tokens.

Examples conceptually:

```text
<BOS>   beginning of sequence
<EOS>   end of sequence
<PAD>   padding
<UNK>   unknown
```

Chat models may also have internal tokens related to:

```text
system
user
assistant
tool messages
```

These special tokens can consume context space too.

So exact API token accounting can include more than visible user text.

## 29. Padding

Suppose batch contains:

```text
Sentence A = 10 tokens
Sentence B = 6 tokens
Sentence C = 8 tokens
```

To build one rectangular tensor, models may pad shorter examples:

```text
A: 10 real tokens

B: 6 real + 4 PAD

C: 8 real + 2 PAD
```

Tensor: $3\times10$

Padding is mainly relevant during batching/training/inference internals.

## 30. Attention masks

Padding tokens generally should not participate as normal content.

Therefore an attention mask might be:

```text
Tokens:

[Hello, world, PAD, PAD]

Mask:

[1, 1, 0, 0]
```

where:

```text
1 = attend
0 = ignore padding
```

This is more transformer-level than RAG-specific, but worth knowing.

## 31. Why different models tokenize differently

The same text:

```text
Retrieval-Augmented Generation
```

can produce different token sequences because models may use:

- different training corpora
- different vocabulary sizes
- different algorithms
- different merge rules
- different byte handling

Therefore:

> Count tokens using the tokenizer corresponding to the model you are actually using.

Do not assume token count from another tokenizer is exact.

### Experiment (added): three tokenizers, same text

In [6]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"

import tiktoken
from transformers import AutoTokenizer

# GPT-4 style BPE, newer GPT-4o style BPE, and BERT-style WordPiece (all-MiniLM-L6-v2)
cl100k = tiktoken.get_encoding("cl100k_base")
o200k = tiktoken.get_encoding("o200k_base")
wordpiece = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

text = "Unbelievable: tokenization splits unhappiness differently."

# Show the pieces each tokenizer produces
print("cl100k   :", [cl100k.decode([i]) for i in cl100k.encode(text)])
print("o200k    :", [o200k.decode([i]) for i in o200k.encode(text)])
print("wordpiece:", wordpiece.tokenize(text))

cl100k   : ['Un', 'belie', 'vable', ':', ' token', 'ization', ' splits', ' unh', 'appiness', ' differently', '.']
o200k    : ['Un', 'bel', 'ievable', ':', ' token', 'ization', ' splits', ' unh', 'appiness', ' differently', '.']
wordpiece: ['unbelievable', ':', 'token', '##ization', 'splits', 'un', '##ha', '##pp', '##iness', 'differently', '.']


Same sentence, three different token lists and counts. Always count tokens with the tokenizer of the model you will actually call.

## 32. Practical token counting

For OpenAI-style tokenization experiments, a common Python library is `tiktoken`.

```bash
pip install tiktoken
```

Example:

In [7]:
# Import tiktoken.
import tiktoken

# Load an encoding.
encoding = tiktoken.get_encoding("cl100k_base")

# Define sample text.
text = "Retrieval-Augmented Generation is useful."

# Convert text into token IDs.
token_ids = encoding.encode(text)

# Display token IDs.
print(token_ids)

# Display the number of tokens.
print(len(token_ids))

[12289, 7379, 838, 62735, 28078, 24367, 374, 5505, 13]
9


For production, use the tokenizer/encoding appropriate to the model you're actually calling.

## 33. Decode tokens

In [8]:
# Decode the token IDs back into text.
decoded_text = encoding.decode(token_ids)

# Display the reconstructed text.
print(decoded_text)

Retrieval-Augmented Generation is useful.


Pipeline:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    n0["Text"] --> n1["encode()"] --> n2["Token IDs"] --> n3["decode()"] --> n4["Text"]
```

## 34. Inspect individual tokens

In [9]:
# Iterate through each token ID.
for token_id in token_ids:

    # Decode one token at a time.
    token_text = encoding.decode([token_id])

    # Display its ID and textual representation.
    print(token_id, repr(token_text))

12289 'Ret'
7379 'rie'
838 'val'
62735 '-Aug'
28078 'mented'
24367 ' Generation'
374 ' is'
5505 ' useful'
13 '.'


This is a useful learning exercise because you'll see things such as:

```text
" Retrieval"
"-"
"Aug"
"mented"
```

depending on the tokenizer.

## 35. Why spaces can be part of tokens

A tokenizer may distinguish:

```text
"hello"
```

from:

```text
" hello"
```

because whitespace carries useful statistical information.

Therefore you may see tokens conceptually like:

```text
" the"
" and"
" model"
```

rather than:

```text
"the"
"and"
"model"
```

This is normal.

## 36. Tokenization and chunking

Suppose The chunking strategy from 04.01 Document Chunking says:

```math
\text{chunk size}=500\,\text{tokens}
```

You should ideally:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    n0["Paragraphs"] --> n1["Sentences"] --> n2["Accumulate<br/>sentences"] --> n3["Count tokens"] --> n4["Stop near 500"]
```

rather than:

```text
take exactly every 500 tokens
```

because exact token boundaries can cut sentences.

## 37. Better token-aware sentence chunking

Architecture:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart TD
    S["Next sentence"] --> C["Count its tokens<br/>(e.g. 40, 55, 85)"]
    C --> D{"Running total<br/>would exceed 500?"}
    D -->|No| A["Add sentence<br/>to current chunk"] --> S
    D -->|Yes| F["Finalize chunk,<br/>start a new one"] --> S
```

This gives:

```text
semantic sentence boundaries
+
token budget control
```

## 38. Token-aware sentence chunker

Setup (added): download the sentence splitter data once.

In [10]:
import nltk

# Sentence-boundary model used by sent_tokenize
nltk.download("punkt_tab", quiet=True)

In [11]:
# Import sentence tokenization.
from nltk.tokenize import sent_tokenize

# Import tiktoken.
import tiktoken

# Load an encoding.
encoding = tiktoken.get_encoding("cl100k_base")

Separate counting function:

In [12]:
# Define a function that counts tokens.
def count_tokens(text, encoding):

    # Encode the text into token IDs.
    token_ids = encoding.encode(text)

    # Return the number of tokens.
    return len(token_ids)

## 39. Group sentences using token limits

In [13]:
# Define a function that groups sentences under a token limit.
def chunk_sentences_by_token_limit(
    text,
    max_tokens,
    encoding,
):

    # Split the document into sentences.
    sentences = sent_tokenize(text)

    # Create the final chunk collection.
    chunks = []

    # Store sentences belonging to the current chunk.
    current_sentences = []

    # Track tokens currently used.
    current_token_count = 0

    # Process every sentence.
    for sentence in sentences:

        # Count tokens in the sentence.
        sentence_tokens = count_tokens(
            sentence,
            encoding,
        )

        # Determine whether adding the sentence exceeds the budget.
        exceeds_limit = (
            current_sentences
            and current_token_count + sentence_tokens > max_tokens
        )

        # Finalize the chunk when the limit would be exceeded.
        if exceeds_limit:

            # Join current sentences.
            chunk = " ".join(current_sentences)

            # Store the completed chunk.
            chunks.append(chunk)

            # Reset the current sentence collection.
            current_sentences = []

            # Reset token count.
            current_token_count = 0

        # Add the sentence to the active chunk.
        current_sentences.append(sentence)

        # Update token usage.
        current_token_count += sentence_tokens

    # Store any remaining sentences.
    if current_sentences:

        # Join remaining sentences.
        chunk = " ".join(current_sentences)

        # Store the final chunk.
        chunks.append(chunk)

    # Return all chunks.
    return chunks

Example (added):

In [14]:
document = ("RAG retrieves evidence before generating. Chunk size is measured in tokens, not words. "
            "An embedding model silently truncates long inputs. Truncated text never reaches the index. "
            "Sentence-aware chunking keeps ideas whole. Token limits decide where a chunk must stop.")

# Group the sentences into chunks of at most 25 tokens
for chunk in chunk_sentences_by_token_limit(document, 25, encoding):

    # Show each chunk with its token count
    print(count_tokens(chunk, encoding), "|", chunk)

17 | RAG retrieves evidence before generating. Chunk size is measured in tokens, not words.
25 | An embedding model silently truncates long inputs. Truncated text never reaches the index. Sentence-aware chunking keeps ideas whole.
9 | Token limits decide where a chunk must stop.


This is much better RAG design than blind token slicing.

## 40. What if one sentence exceeds the limit?

Suppose:

```math
\text{sentence}=900\,\text{tokens}
```

and:

```math
\text{max\_chunk}=500
```

Sentence-aware splitting alone fails.

Use fallback:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    S["Sentence"] --> Q{"Too large?"}
    Q -->|Yes| P["Split by smaller<br/>linguistic or token<br/>boundaries"]
    Q -->|No| K["Keep whole"]
```

That's exactly why recursive chunking from 04.01 Document Chunking is useful.

## 41. Tokenization and embeddings

Embedding models also have maximum input lengths.

Suppose embedding model accepts: $M$

tokens.

Chunk: $C\gt M$

Then depending on implementation you might see:

```text
error
```

or worse:

```text
silent truncation
```

## 42. Why silent truncation is dangerous

> [!TIP]
> **In simple words:** If a chunk is longer than the embedding model's limit, the end is ignored and that part of the document can never be found.

Suppose chunk:

```text
Token 1
...
Token 500
...
Token 1000
...
IMPORTANT ANSWER at token 1500
```

Embedding model only processes first: $1000$

tokens.

Then:

```text
important answer
```

never contributes to the vector.

Yet your database stores the full chunk text.

So you may think:

> Why isn't retrieval finding this sentence? It exists in the chunk.

Because:

> It may not have existed in the embedding input.

Very important debugging insight.

## 43. Always validate embedding-token length

Conceptually:

Setup (added): a chunk that is too long for a 128-token embedding model.

In [15]:
# A long chunk and a small model limit
chunk_text = "Ingestion, chunking and embedding must agree on token limits. " * 20
embedding_limit = 128

In [16]:
# Count tokens before embedding.
token_count = count_tokens(
    chunk_text,
    encoding,
)

# Reject or split oversized chunks.
if token_count > embedding_limit:
    print("Chunk must be split.")

Chunk must be split.


Do not rely blindly on truncation.

## 44. Tokenization and multilingual RAG

This is particularly important.

Suppose English sentence:

```text
Artificial intelligence improves search.
```

Perhaps produces roughly a small number of tokens.

Equivalent content in another language might tokenize into significantly more or fewer tokens depending on tokenizer vocabulary.

So:

```math
\text{tokens per semantic unit}
```

can vary substantially between languages.

## 45. Why multilingual token efficiency matters

Suppose:

```text
English document
500 words → 650 tokens
```

while another language might hypothetically require:

```text
500 words-equivalent → 1100 tokens
```

Then a fixed:

```text
500-token chunk
```

contains much less semantic content in the second language.

This can impact:

- retrieval granularity
- number of chunks
- storage
- embedding costs
- context capacity

### Experiment (added): the same sentence in five languages

In [17]:
sentences = {
    "English": "The customer can return the product within thirty days for a full refund.",
    "German": "Der Kunde kann das Produkt innerhalb von dreißig Tagen gegen volle Rückerstattung zurückgeben.",
    "Spanish": "El cliente puede devolver el producto dentro de treinta días para un reembolso completo.",
    "Hindi": "ग्राहक पूर्ण धनवापसी के लिए तीस दिनों के भीतर उत्पाद वापस कर सकता है।",
    "Japanese": "お客様は全額返金のために30日以内に商品を返品できます。",
}

for language, sentence in sentences.items():
    # Token counts for the same meaning in each language
    print(f"{language:<9} cl100k = {len(cl100k.encode(sentence)):>3}   o200k = {len(o200k.encode(sentence)):>3}")

English   cl100k =  14   o200k =  14
German    cl100k =  25   o200k =  19
Spanish   cl100k =  20   o200k =  16
Hindi     cl100k =  67   o200k =  20
Japanese  cl100k =  30   o200k =  19


```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
xychart-beta
    title "Tokens for one sentence (cl100k)"
    x-axis ["English", "German", "Spanish", "Hindi", "Japanese"]
    y-axis "tokens" 0 --> 70
    bar [14, 25, 20, 67, 30]
```

Same meaning, very different token counts. A "500-token chunk" holds much less Hindi text than English text with this tokenizer; the newer `o200k` vocabulary narrows the gap.

## 46. Therefore multilingual chunking should be token-aware

Avoid:

```text
500 English words
500 Japanese "words"
500 German words
```

as if they were comparable.

Instead evaluate:

```text
token length
+
semantic boundaries
+
language
```

## 47. German compound words

For regulatory RAG, imagine German:

```text
Batterierecht-Durchführungsgesetz
```

A tokenizer may split this into multiple meaningful/unfamiliar subwords.

This can affect exact-match and dense retrieval behavior.

Similarly very long chemical/product/legal identifiers can consume several tokens.

## 48. Tokenization and exact identifiers

Consider:

```text
EU 2023/1542
```

or:

```text
G.S.R. 180(E)
```

or:

```text
ISO/IEC 27001:2022
```

The tokenizer may fragment these into several tokens.

Dense embeddings may still understand some semantics, but exact identifiers are another reason hybrid search/BM25 is valuable.

Preserve these strings exactly during ingestion.

### Experiment (added): how identifiers are split

In [18]:
for text in ["E401", "error E-401", "6ES7513-1AL02-0AB0", "Python 3.12", "calculate_customer_refund_amount()"]:
    # BPE pieces for each identifier
    pieces = [cl100k.decode([i]) for i in cl100k.encode(text)]

    print(f"{text!r:<38} {len(pieces):>2} tokens  {pieces}")

'E401'                                  2 tokens  ['E', '401']
'error E-401'                           4 tokens  ['error', ' E', '-', '401']
'6ES7513-1AL02-0AB0'                   12 tokens  ['6', 'ES', '751', '3', '-', '1', 'AL', '02', '-', '0', 'AB', '0']
'Python 3.12'                           5 tokens  ['Python', ' ', '3', '.', '12']
'calculate_customer_refund_amount()'    6 tokens  ['calculate', '_customer', '_ref', 'und', '_amount', '()']


Part numbers and function names break into many small pieces. That is why exact identifiers are better handled by BM25 (05 Sparse Retrieval) than by embeddings alone.

## 49. Tokenization and code

Code may tokenize inefficiently because it contains:

```text
underscores
camelCase
parentheses
operators
indentation
long variable names
```

Example:

```python
calculate_customer_refund_amount()
```

could be split into several tokens.

Therefore:

```math
500\,\text{tokens of prose}
```

and:

```math
500\,\text{tokens of code}
```

may represent very different amounts of semantic content.

Again:

> Chunk by semantic unit first, token budget second.

## 50. Tokenization and cost

Suppose provider pricing is based on tokens.

Conceptually:

```math
\text{Cost} = T_{\text{input}}\times P_{\text{input}} + T_{\text{output}}\times P_{\text{output}}
```

where:

- $T_{input}$ = input tokens
- $T_{output}$ = generated tokens
- $P$ = price/token

For RAG, retrieved chunks can become the largest part of input cost.

## 51. Example cost comparison

Suppose each query retrieves: $10$

chunks.

Each: $500$

tokens.

Retrieval context: $5000$

tokens.

If 100,000 user questions are asked:

```math
5000\times100,000 = 500,000,000
```

retrieval-context tokens alone.

This is why context optimization matters financially.

## 52. Chunk overlap increases token cost twice

Overlap causes extra cost during:

### Indexing

You embed duplicated text.

### Querying

Multiple overlapping chunks may be retrieved and inserted into the prompt.

Example:

```text
Chunk A:
A B C D

Chunk B:
C D E F
```

Prompt includes:

```text
C D
```

twice.

So overlap can increase:

```text
embedding cost
+
LLM inference cost
```

## 53. RAG token-budget architecture

A production context builder might explicitly allocate:

```text
Total context = 128K

System                     4K
Conversation              15K
Query                       1K
Retrieved evidence         70K
Tool results               10K
Output reserve             20K
Safety reserve              8K
```

You don't necessarily hardcode these exact numbers, but designing with explicit budgets is valuable.

## 54. Dynamic token budgeting

> [!TIP]
> **In simple words:** Work out how many tokens are left for evidence after fixed parts, then fill only that space.

Suppose user has no chat history.

Then:

```text
history = 0
```

More room for retrieval.

If user has long history:

```text
history = 12,000
```

retrieval budget should shrink or history should be summarized.

Conceptually:

```math
T_{\text{retrieval}} = T_{\text{context}} - T_{\text{fixed}} - T_{\text{history}} - T_{\text{outputReserve}}
```

This should often be dynamic.

## 55. Context packing

Suppose candidate chunks:

```text
C1 = 600 tokens
C2 = 450
C3 = 900
C4 = 300
C5 = 800
```

Available budget: $2000$

You cannot blindly send all five.

Potential strategy:

```text
C1 + C2 + C4
=
600 + 450 + 300
=
1350
```

Then potentially another chunk if it fits.

Context packing becomes a kind of constrained optimization problem.

## 56. Simple greedy packing

In [19]:
# Define a function for packing ranked chunks under a token budget.
def pack_chunks(
    ranked_chunks,
    token_budget,
    encoding,
):

    # Create a list for selected chunks.
    selected_chunks = []

    # Track tokens already used.
    used_tokens = 0

    # Process chunks in ranking order.
    for chunk in ranked_chunks:

        # Count tokens in the current chunk.
        chunk_tokens = count_tokens(
            chunk,
            encoding,
        )

        # Check whether the chunk fits in the remaining budget.
        if used_tokens + chunk_tokens <= token_budget:

            # Keep the chunk.
            selected_chunks.append(chunk)

            # Update token usage.
            used_tokens += chunk_tokens

    # Return selected chunks and final token usage.
    return selected_chunks, used_tokens

Example (added):

In [20]:
ranked = ["Password resets are done from the login page under Forgot password.",
          "Reset links expire after 30 minutes and can be requested again.",
          "The office is closed on public holidays.",
          "Accounts lock after five failed login attempts and unlock after 15 minutes."]

# Keep the best-ranked chunks that fit in 40 tokens
selected, used = pack_chunks(ranked, 40, encoding)

print("used tokens:", used)

for s in selected:
    print("-", s)

used tokens: 33
- Password resets are done from the login page under Forgot password.
- Reset links expire after 30 minutes and can be requested again.
- The office is closed on public holidays.


This is simplistic but captures the idea.

Production context selection may use:

```text
ranking score
diversity
parent relationships
token cost
answerability
```

## 57. Token budgeting vs Top-K

A naive retriever says:

```text
Top K = 10
```

A better architecture can say:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    n0["Retrieve Top 50"] --> n1["Rerank"] --> n2["Pack best evidence<br/>within an<br/>8,000-token budget"]
```

This is often stronger than fixed Top-K.

So later you'll see: $\text{TopK}$

replaced conceptually by:

```math
\text{Best Evidence Under Token Budget}
```

## 58. Tokenizer vocabulary size trade-off

Suppose vocabulary is larger.

Advantages:

```text
common words require fewer tokens
shorter sequences
```

But embedding/output matrices become larger.

Roughly, token embedding parameter count: $|V|\times d$

If: $|V|=100,000$

and: $d=4096$

then:

```math
100,000\times4096 = 409,600,000
```

parameters just for one embedding matrix representation, conceptually.

Large vocabularies aren't free.

## 59. Small vocabulary trade-off

Smaller vocabulary:

```text
fewer vocabulary entries
```

but words get split more often:

```text
electromagnetic
→
elect
ro
mag
netic
```

Sequence becomes longer.

So tokenizer design balances:

```math
\text{Vocabulary Size} \leftrightarrow \text{Sequence Length}
```

## 60. Compression ratio

A useful tokenizer property is roughly:

```math
\text{Compression} = \frac{\text{characters or bytes}} {\text{tokens}}
```

Higher characters-per-token means shorter token sequences for the same text.

But efficiency differs significantly by:

```text
language
domain
code
identifiers
```

## 61. Why tokenizer choice affects model performance

Suppose a domain repeatedly contains:

```text
electrophotolithography
```

but tokenizer splits it into 15 tokens.

The model needs more positions and must learn relationships between many subpieces.

A tokenizer well matched to the domain might represent it with fewer useful subwords.

This affects:

```text
efficiency
context utilization
sometimes learning quality
```

## 62. Should you create your own tokenizer for enterprise RAG?

Usually **no**.

Why?

Your embedding model and LLM already expect their tokenizer.

Changing tokenizer independently will not change the pretrained model's vocabulary.

You normally:

```text
use model's existing tokenizer
```

rather than training a custom tokenizer for RAG.

Custom tokenization becomes more relevant when training/pretraining your own language model.

## 63. Very important misconception

Don't confuse:

### Tokenizer

```text
text → token IDs
```

with:

### Embedding model

```text
text/tokens → semantic vector
```

with:

### Vector database

```text
semantic vectors → nearest-neighbor search
```

Full pipeline:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart TB
    n0["Text"] --> n1["Tokenizer"] --> n2["Token IDs"] --> n3["Neural embedding<br/>model"] --> n4["Dense vector"] --> n5["Vector database"]
```

## 64. Another misconception

> Tokens are always meaningful words or subwords.

Not necessarily.

They are vocabulary units optimized statistically.

A token could be:

```text
word
subword
punctuation
space-prefixed fragment
byte sequence
special control token
```

## 65. Another misconception

> One token ≈ four characters.

You may see this as a rough English heuristic.

It is **not a rule**.

It varies with:

- tokenizer
- language
- text type
- code
- numbers
- Unicode

Never use it for exact production limits.

Use the actual tokenizer.

## 66. Tokenization failure in RAG: truncation

Imagine:

```text
Chunk length = 1200 tokens
Embedding limit = 1000
```

The final 200 tokens contain the answer.

If truncated:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    n0["Retriever vector"] --> n1["Does not represent<br/>the answer"]
```

Then retrieval recall suffers.

This looks like:

```text
"embedding model is bad"
```

but root cause is actually:

```text
chunk/tokenization configuration
```

## 67. Another failure: wrong token budget assumptions

Suppose application uses:

```text
chunk_size = 1000 characters
```

because developer thinks:

```text
1000 characters = 1000 tokens
```

Wrong.

Depending on content:

```text
1000 characters
```

might produce far fewer or sometimes many more tokens than expected.

This can lead to inconsistent chunk sizes.

## 68. Another failure: multilingual expansion

Suppose the pipeline was tuned using English only.

Production adds Japanese documents.

Suddenly:

```text
more chunks/document
higher embedding cost
different retrieval granularity
context fills faster
```

because tokenization behavior differs.

Therefore evaluation sets should reflect actual production languages.

## 69. Interview question: What is tokenization?

Strong answer:

> Tokenization converts raw text into discrete units called tokens, which are mapped to numerical IDs before being processed by a language model. Modern LLMs generally use subword or byte-aware schemes because they provide a balance between manageable vocabulary size, efficient sequence length, and the ability to represent rare or unseen text.

## 70. Interview question: Why not tokenize by words?

Answer:

> Word-level tokenization requires an enormous vocabulary and has an out-of-vocabulary problem. Subword tokenization allows rare and unseen words to be represented by combinations of reusable components while keeping sequence lengths much shorter than character-level tokenization.

## 71. Interview question: Explain BPE

Strong short answer:

> Byte Pair Encoding starts with small symbols and repeatedly merges frequently occurring adjacent pairs. Over many iterations it learns reusable subwords. Common strings can therefore be represented with few tokens while rare strings remain representable through smaller units.

## 72. Interview question: BPE vs WordPiece

Good answer:

> Both are subword tokenizers. BPE traditionally performs frequent adjacent-pair merges, while WordPiece chooses subword units using a likelihood/scoring objective. Their engineering goal is similar: reduce vocabulary explosion while retaining the ability to represent rare words.

Don't overcomplicate this unless asked.

## 73. Interview question: What is SentencePiece?

> SentencePiece is a tokenizer training framework that works directly on raw text rather than requiring pre-segmented words. It can implement approaches such as BPE or the Unigram model and is particularly useful for multilingual languages where spaces don't reliably indicate word boundaries.

## 74. Interview question: Why does tokenization matter to RAG?

A strong answer:

> Tokenization determines the actual size of chunks seen by embedding models and the amount of retrieved evidence that fits in the LLM context window. It therefore affects truncation, retrieval granularity, context budgeting, inference cost, multilingual behavior, and the maximum amount of evidence that can be passed into generation.

## 75. Interview question: Should chunk size be characters, words, or tokens?

Good answer:

> I prefer semantic boundaries first and token limits second. Tokens are useful for enforcing model constraints, but blindly cutting every N tokens can fragment meaning. A good splitter preserves headings, paragraphs or sentences and uses token counts as minimum/maximum size constraints.

This connects 04.01 Document Chunking and this chapter nicely.

## 76. Interview question: What happens if a chunk exceeds embedding-model context?

Possible outcomes:

```text
request failure
or
truncation
```

The dangerous case is silent truncation because later content may never contribute to the embedding.

Strong production system:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    n0["Count tokens"] --> n1["Validate against<br/>limit"] --> n2["Split before<br/>embedding"]
```

## 77. Interview question: Why can multilingual RAG behave differently?

Because tokenizer efficiency differs between languages.

Equivalent semantic content may generate different token counts, which affects:

```text
chunk size
number of vectors
context utilization
cost
```

Therefore multilingual chunking and evaluation should use actual tokenizer statistics from each production language.

## 78. FAANG interview trap

### Trap: "Token = word."

Wrong.

### Trap: "One token is always four characters."

Wrong.

### Trap: "BPE simply splits words."

Incomplete.

BPE **learns merges** from frequent adjacent units.

### Trap: "SentencePiece is the same algorithm as BPE."

Wrong.

SentencePiece is a tokenization framework capable of using approaches such as BPE or Unigram.

### Trap: "Model context = number of retrieved tokens."

Wrong.

Context includes more than retrieved evidence:

```math
\text{System} + \text{History} + \text{Query} + \text{Retrieved Evidence} + \text{Tool Content} + \text{Generation Reserve}
```

## 79. Production RAG token architecture

A strong system can look like:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart TD
    D["Document"] --> S["Structure-aware<br/>split"] --> SE["Sentences"] --> T{"Token count<br/>within limit?"}
    T -->|Yes| C["Chunk"]
    T -->|No| R["Recursive split"] --> C
    C --> E["Embedding model"]
```

At query time:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart TB
    n0["Query"] --> n1["Retrieve Top 30"] --> n2["Rerank"] --> n3["Token-aware<br/>context packer"] --> n4["Fit best chunks<br/>under the context<br/>budget"] --> n5["LLM"]
```

## 80. Most important equations

### Context limit

```math
T_{\text{input}}+T_{\text{output}}\leq T_{\text{context}}
```

### Retrieval budget

```math
T_{\text{retrieval}} = T_{\text{context}} - T_{\text{system}} - T_{\text{history}} - T_{\text{query}} - T_{\text{outputReserve}}
```

### Approximate inference cost

```math
\text{Cost} = T_{\text{input}}P_{\text{input}} + T_{\text{output}}P_{\text{output}}
```

### Embedding-table parameter intuition

```math
\text{Parameters} \approx |V|\times d
```

## 81. What you need to remember for interviews

The entire chapter can be compressed into this diagram:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart TD
    R["Raw text"] --> T["Tokenizer"]
    T --> V["Vocabulary"]
    T --> RU["Rules"]
    V --> I["Token IDs"]
    RU --> I
    I --> N["Neural network"]
```

For RAG:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart LR
    subgraph IX["Index time"]
        direction TB
        D["Document"] --> S["Semantic<br/>splitting"] --> T["Token count"] --> C["Chunk"] --> E["Embedding"]
    end
    subgraph QT["Query time"]
        direction TB
        Q["Query"] --> R["Retrieval"] --> RC["Retrieved<br/>chunks"] --> B["Token budget"] --> P["Context<br/>packing"] --> L["LLM"]
    end
    style IX fill:#EFF6FF,stroke:#2563EB
    style QT fill:#F0FDF4,stroke:#16A34A
```

## 82. Revision sheet

Remember these **10 things**:

1. **Token ≠ word.**
2. Models operate on **token IDs**, not raw strings.
3. Modern tokenizers commonly use **subword/byte-aware techniques**.
4. BPE learns frequently occurring adjacent merges.
5. WordPiece is another subword strategy.
6. SentencePiece works directly on raw text and can use algorithms such as BPE or Unigram.
7. Different models can tokenize the same sentence differently.
8. Token counts determine chunk limits and context-window usage.
9. Never allow embedding-model truncation to silently destroy useful chunk content.
10. For RAG:

```math
\boxed{\text{semantic boundaries first, token limits second}}
```

That's the most important practical takeaway.

## 83. How this chapter connects

You now understand:

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
flowchart TB
    n0["02.01 RAG<br/>Fundamentals: why<br/>RAG exists"] --> n1["03 Data Ingestion:<br/>raw documents<br/>become structured<br/>knowledge"] --> n2["04.01 Document<br/>Chunking:<br/>documents become<br/>retrievable chunks"] --> n3["01.02<br/>Tokenization:<br/>chunks within<br/>model token limits"] --> n4["01.04 Embeddings<br/>Fundamentals"]
```

**01.04 Embeddings Fundamentals** is a must-master topic and goes deeper than this chapter. It covers what embeddings really represent, semantic spaces, dimensions, query/document embeddings, bi-encoders, pooling, similarity, why similar concepts become nearby vectors, limitations of embeddings, model selection, multilingual/domain embeddings, and the connection from token-level transformer representations to a single document vector.

## 84. Practice Tasks

1. Count words, characters and tokens for five paragraphs of your own documents. What is your characters-per-token ratio?
2. Run the BPE functions for five merge iterations on `["low", "lower", "newest", "widest"]` and print each merge.
3. Tokenize ten product codes from your domain with `cl100k_base` and `o200k_base`; which tokenizer is more compact?
4. Find your embedding model's maximum input length and write a check that rejects longer chunks.
5. Change `chunk_sentences_by_token_limit` so a single sentence longer than the limit is split by tokens.
6. Compute a token budget for a 16,000-token model with a 1,000-token system prompt, 3,000 tokens of history and a 2,000-token answer reserve.
7. Measure how many tokens the same FAQ costs in two languages you support.

## 85. Final Takeaways

```mermaid
%%{init: {'theme':'base','themeVariables':{'primaryColor':'#EEF2FF','primaryBorderColor':'#6366F1','primaryTextColor':'#1E1B4B','lineColor':'#6366F1'}}}%%
timeline
    title Tokens and tokenization at a glance
    Units : token is not a word : models read token IDs
    Methods : BPE merges frequent pairs : WordPiece uses ## : SentencePiece handles raw text
    Limits : chunk size in tokens : embedding input limit : context window
    Budget : system + history + query + evidence + answer
    Risks : silent truncation : multilingual token growth : split identifiers
    Rule : semantic boundaries first, token limits second
```

> [!TIP]
> **Study rule:** whenever a number in RAG is a "size" or a "limit", ask which tokenizer counted it.